In [ ]:
import ee
import subprocess
from google.cloud import storage
import os

This script uploads rasters from the bucket to GEE as assets.

In [ ]:
# provide access key

In [ ]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/credentials.json"

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='your-cloud-project')

In [ ]:
!earthengine set_project your-cloud-project

## 0. If you want to upload only one specific image

In [ ]:
# upload
#! earthengine upload image --asset_id=projects/your-cloud-project/assets/test/test_asset gs://bucket_name/baltic_wq_ml/era5_monthly/tmp/bal_era5_total_precipitation_sum_monthly_20231201.tif

## 1. Set variables

In [ ]:
region_name = 'estonia'

In [ ]:
bucket_name = "bucket_name"
gcs_folder = f"{region_name}_working/era5_monthly_rasters_substep/"  # The folder inside the bucket containing the images
gee_folder = f"projects/your-cloud-project/assets/era5_monthly_rasters_{region_name}"  # GEE destination folder

In [ ]:
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

parameters = ['total_evaporation_sum', 'total_precipitation_sum', 'maximum_2m_temperature', 'mean_2m_temperature', 'minimum_2m_temperature']

In [ ]:
# Create the folder where rasters will be stored

In [ ]:
! earthengine create folder projects/your-cloud-project/assets/era5_monthly_rasters_europe

## 2. Create folders in GEE assets

In [ ]:
# OPTIONAL STEP - delete an existing folder,
# as overwriting isn't possible

def delete_folder_and_assets(folder_path):
    """Deletes a folder and all its assets."""
    # List all assets in the folder
    assets = list_assets(folder_path)
    
    # Delete each asset
    for asset in assets:
        if asset:  # Ensure the asset string is not empty
            print(f"Deleting asset: {asset}")
            delete_asset(asset)
    
    # Delete the folder itself
    print(f"Deleting folder: {folder_path}")
    delete_asset(folder_path)
    
def list_assets(folder_path):
    """Lists all assets in the folder."""
    result = subprocess.run(
        ["earthengine", "ls", "--recursive", folder_path],
        stdout=subprocess.PIPE,
        text=True
    )
    return result.stdout.strip().split('\n')

def delete_asset(asset_path):
    """Deletes a single asset."""
    subprocess.run(["earthengine", "rm", asset_path], check=True)

In [ ]:
def create_folder(path):
    """Creates all folders in the provided path, ignore if exists."""
    
    parts = path.split('/')
    current_path = gee_folder
    
    for part in parts:
        current_path = f"{current_path}/{part}" if current_path else part
        try:
            # Attempt to create the folder
            subprocess.run(
                ["earthengine", "create", "folder", current_path],
                check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
            )
            print(f"Created folder: {current_path}")
        except subprocess.CalledProcessError as e:
            # Handle cases where the folder already exists
            if "already exists" in e.stderr.decode():
                print(f"Folder already exists: {current_path}")
            else:
                print(f"Failed to create folder: {current_path}. Error: {e.stderr.decode()}")


## 3. Upload folder contents from the bucket

In [ ]:
def list_gcs_files(bucket_name, gcs_folder):
    """
    List all files in a GCS folder.
    """
    client = storage.Client()
    bucket = client.get_bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=gcs_folder)
    return [blob.name for blob in blobs if not blob.name.endswith('/')]

In [ ]:
def upload_to_gee(file_paths, gee_folder):

    for file_path in file_paths:
        # Extract the base name of the image (e.g., image.png)
        image_name = file_path.split("/")[-1]
        
        parameter = file_path.split("/")[2]
        
        year = file_path.split("/")[3]
        
        
        # Construct the GEE asset ID (remove file extension)
        asset_id = f"{gee_folder}/{parameter}/{year}/{image_name.rsplit('.', 1)[0]}"
        
        # Full GCS path
        gcs_path = f"gs://{bucket_name}/{file_path}"
        
        # Run the earthengine upload command via subprocess
        command = [
            "earthengine", 
            "upload", 
            "image", 
            f"--asset_id={asset_id}", 
            gcs_path
        ]
        try:
            result = subprocess.run(command, check=True, capture_output=True, text=True)
            print(f"Started upload for {image_name} to {asset_id}.\nOutput:\n{result.stdout}")
        except subprocess.CalledProcessError as e:
            print(f"Failed to upload {image_name}.\nError:\n{e.stderr}")

In [ ]:
### function calls

In [ ]:
# before uploading the assets, we need to make sure that there are no such assets, as overwriting is not possible
#delete_folder_and_assets('projects/your-cloud-project/assets/era5_geotiffs_2020_2023')

In [ ]:
# Create folders of years and parameters where to store assets

for year in years:
    
    for parameter in parameters:
        
        path = f"{parameter}/{year}"
        
        create_folder(path)

In [ ]:
# List files in GCS folder
files = list_gcs_files(bucket_name, gcs_folder)

In [ ]:
print(files)

In [ ]:
# optional - upload only specific parameter/year..

#files_to_upload = [f for f in files if str(years[0]) in f]

In [ ]:
#files_to_upload

In [ ]:
# Upload files to GEE

upload_to_gee(files, gee_folder)